# Experimentos

Este proyecto compara tres algoritmos de aprendizaje automático (dos de clasificación y uno de regresión)
usando un diseño experimental 3×3×3 sobre los hiperparámetros principales.
El objetivo es evaluar rendimiento sin alto costo computacional.

**Algoritmos:**
- LinearRegression (regresión)
- LogisticRegression (clasificación)
- DecisionTreeClassifier (clasificación)

**Estructura de datos:**
- Train / Validation / Test


In [10]:
# ==========================================
# 1. Importación de librerías y carga de datos
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# Cargar datos ya preparados desde el notebook Prep.ipynb
df = pd.read_csv("../Data/video_game_reviews.csv")

# Vista general
df.head()


,Game Title,User Rating,Age Group Targeted,Price,Platform,Requires Special Device,Developer,Publisher,Release Year,Genre,Multiplayer,Game Length (Hours),Graphics Quality,Soundtrack Quality,Story Quality,User Review Text,Game Mode,Min Number of Players
0,Grand Theft Auto V,36.4,All Ages,41.41,PC,No,Game Freak,Innersloth,2015,Adventure,No,55.3,Medium,Average,Poor,"Solid game, but too many bugs.",Offline,1
1,The Sims 4,38.3,Adults,57.56,PC,No,Nintendo,Electronic Arts,2015,Shooter,Yes,34.6,Low,Poor,Poor,"Solid game, but too many bugs.",Offline,3
2,Minecraft,26.8,Teens,44.93,PC,Yes,Bungie,Capcom,2012,Adventure,Yes,13.9,Low,Good,Average,"Great game, but the graphics could be better.",Offline,5
3,Bioshock Infinite,38.4,All Ages,48.29,Mobile,Yes,Game Freak,Nintendo,2015,Sports,No,41.9,Medium,Good,Excellent,"Solid game, but the graphics could be better.",Online,4
4,Half-Life: Alyx,30.1,Adults,55.49,PlayStation,Yes,Game Freak,Epic Games,2022,RPG,Yes,13.2,High,Poor,Good,"Great game, but too many bugs.",Offline,1


In [11]:
# ==========================================
# 2. Definición de variables
# ==========================================

# Variable dependiente (objetivo)
target = 'User Rating'

# Variables independientes (todas menos la de salida)
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target]

# División del dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Variables independientes:", features[:5], "...")
print("Variable dependiente:", target)


Variables independientes: ['Game Title', 'Age Group Targeted', 'Price', 'Platform', 'Requires Special Device'] ...
Variable dependiente: User Rating


In [12]:
# ==========================================
# 2.5. Codificación de variables categóricas
# ==========================================

# Detectar columnas no numéricas
cat_cols = X.select_dtypes(include=['object', 'category']).columns
print("Columnas categóricas:", list(cat_cols))

# One-Hot Encoding automático (ligero)
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Actualizar división en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print(f"Nuevo tamaño del dataset codificado: {X_encoded.shape}")


Columnas categóricas: ['Game Title', 'Age Group Targeted', 'Platform', 'Requires Special Device', 'Developer', 'Publisher', 'Genre', 'Multiplayer', 'Graphics Quality', 'Soundtrack Quality', 'Story Quality', 'User Review Text', 'Game Mode']
Nuevo tamaño del dataset codificado: (47774, 99)


In [13]:
# ==========================================
# 3. Escalamiento de datos
# ==========================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Datos escalados correctamente")
print(f"Shape X_train_scaled: {X_train_scaled.shape}")
print(f"Shape X_test_scaled: {X_test_scaled.shape}")


Datos escalados correctamente
Shape X_train_scaled: (38219, 99)
Shape X_test_scaled: (9555, 99)


In [14]:
# ==========================================
# 4. Definición de modelos y sus hiperparámetros (OPTIMIZADO)
# ==========================================

# Grid reducido: 2×2×2 = 8 combinaciones en lugar de 27
# Esto reduce el tiempo de ejecución de ~45 min a ~10 min

modelos = {
    "Regresión Lineal": {
        "modelo": LinearRegression(),
        "param_grid": {
            "fit_intercept": [True, False],
            "positive": [False, True]
        }
    },
    "Random Forest": {
        "modelo": RandomForestRegressor(random_state=42, n_jobs=-1),
        "param_grid": {
            "n_estimators": [50, 100],           # Reducido de 3 a 2 valores
            "max_depth": [10, 15],               # Reducido: eliminamos 5
            "min_samples_split": [2, 5]          # Reducido: eliminamos 10
        }
    },
    "XGBoost": {
        "modelo": XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1),
        "param_grid": {
            "n_estimators": [50, 100],           # Reducido de 3 a 2 valores
            "learning_rate": [0.05, 0.1],        # Reducido: eliminamos 0.2
            "max_depth": [3, 6]                  # Reducido: eliminamos 9
        }
    }
}


In [15]:
# ==========================================
# 5. Entrenamiento y evaluación con GridSearchCV
# ==========================================

resultados = []

for nombre, info in modelos.items():
    print(f"\n{'='*50}")
    print(f"Entrenando modelo: {nombre}")
    print(f"{'='*50}")
    
    grid = GridSearchCV(info["modelo"], info["param_grid"],
                        scoring='neg_mean_absolute_error', cv=3, 
                        n_jobs=-1, verbose=2)  # verbose=2 muestra progreso
    grid.fit(X_train_scaled, y_train)
    
    # Mejor modelo
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    resultados.append({
        "Modelo": nombre,
        "Mejores parámetros": grid.best_params_,
        "MAE": mae,
        "RMSE": rmse
    })
    
    print(f"\n {nombre} completado!")
    print(f"  MAE: {mae:.4f}, RMSE: {rmse:.4f}")

print("\n" + "="*50)
print("Entrenamiento finalizado.")
print("="*50)



Entrenando modelo: Regresión Lineal
Fitting 3 folds for each of 4 candidates, totalling 12 fits

 Regresión Lineal completado!
  MAE: 1.0007, RMSE: 1.1581

Entrenando modelo: Random Forest
Fitting 3 folds for each of 8 candidates, totalling 24 fits

 Random Forest completado!
  MAE: 1.0128, RMSE: 1.1803

Entrenando modelo: XGBoost
Fitting 3 folds for each of 8 candidates, totalling 24 fits

 XGBoost completado!
  MAE: 1.0052, RMSE: 1.1666

Entrenamiento finalizado.


In [16]:
# ==========================================
# 5. Entrenamiento y evaluación con GridSearchCV
# ==========================================

resultados = []

for nombre, info in modelos.items():
    print(f"Entrenando modelo: {nombre}")
    
    grid = GridSearchCV(info["modelo"], info["param_grid"],
                        scoring='neg_mean_absolute_error', cv=3, n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    # Mejor modelo
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    resultados.append({
        "Modelo": nombre,
        "Mejores parámetros": grid.best_params_,
        "MAE": mae,
        "RMSE": rmse
    })

print("Entrenamiento finalizado.")


Entrenando modelo: Regresión Lineal
Entrenando modelo: Random Forest
Entrenando modelo: XGBoost
Entrenamiento finalizado.


In [17]:
# ==========================================
# 6. Tabla comparativa de resultados
# ==========================================

tabla_resultados = pd.DataFrame(resultados)
tabla_resultados = tabla_resultados.sort_values(by="RMSE")
tabla_resultados.reset_index(drop=True, inplace=True)
tabla_resultados


,Modelo,Mejores parámetros,MAE,RMSE
0,Regresión Lineal,"{'fit_intercept': True, 'positive': True}",1.000656,1.158090
1,XGBoost,"{'learning_rate': 0.05, 'max_depth': 6, 'n_est...",1.005163,1.166599
2,Random Forest,"{'max_depth': 10, 'min_samples_split': 5, 'n_e...",1.012815,1.180342


## Conclusiones del Experimento

### Mejor Modelo

El modelo con mejor desempeño fue la **Regresión Lineal** con los siguientes parámetros:
- `fit_intercept`: True
- `positive`: True

**Métricas de rendimiento:**
- MAE: 1.0007
- RMSE: 1.1581

### Análisis Comparativo

1. **Regresión Lineal**: Sirve como línea base y resultó ser el mejor modelo en este caso. Esto sugiere que la relación entre las variables es predominantemente lineal.

2. **Random Forest**: Ofrece un buen equilibrio entre precisión y generalización. Su desempeño fue muy cercano al modelo lineal (MAE: 1.0128, RMSE: 1.1803).

3. **XGBoost**: Aunque suele ser el más preciso para relaciones complejas, en este dataset obtuvo un rendimiento intermedio (MAE: 1.0052, RMSE: 1.1666). El costo computacional fue medio.

### Recomendaciones para Mejorar

- Ajustar el preprocesamiento de datos para capturar mejor las características relevantes
- Eliminar ruido o valores atípicos que puedan afectar el rendimiento
- Aumentar el tamaño del dataset si es posible
- Realizar feature engineering para crear variables más informativas
- Probar con validación cruzada para mayor robustez en la evaluación
